# 🦕 Dino SDK v1.1.0 - Detecção de Contexto Aprimorada

## 🚀 Nova Abordagem: Extração Robusta de Token e URL

O Dino SDK v1.1.0 implementa **detecção robusta de contexto** do notebook Databricks usando múltiplos métodos:

### ✅ **Principais Melhorias:**
- **Extração nativa** de token via `dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()`
- **Detecção automática** de workspace URL via `dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()`
- **Múltiplos fallbacks** - dbutils globais, DBUtils importado, Spark context, env vars
- **Logs detalhados** para diagnóstico completo

### 🔧 **Problema Resolvido:**
```
❌ Erro ao criar token temporário: ❌ Não foi possível determinar a URL do workspace
⚠️ dbutils não disponível
⚠️ DATABRICKS_HOST/DATABRICKS_TOKEN não encontrados
```

**Solução**: Extrair token e URL **diretamente do contexto do notebook** via dbutils.

## 1. 🔍 Diagnóstico Completo do Ambiente

Vamos verificar **todos** os métodos de detecção de contexto disponíveis no ambiente.

In [ ]:
# Diagnóstico completo de contexto Databricks
import os
import logging

print("🔍 Diagnóstico Completo de Contexto Databricks v1.1.0")
print("=" * 60)

# Configurar logging para ver detalhes
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# 1. Verificar dbutils em globals
print("1️⃣ Verificar dbutils em globals:")
if 'dbutils' in globals():
    dbutils = globals()['dbutils']
    print("   ✅ dbutils encontrado em globals")
    
    # Tentar extrair token e URL
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        
        # API URL
        try:
            api_url = context.apiUrl().get() if context.apiUrl().isDefined() else None
            if api_url:
                print(f"   ✅ Workspace URL: {api_url}")
                # Armazenar para testes posteriores
                globals()['detected_workspace_url'] = f"https://{api_url}" if not api_url.startswith('https://') else api_url
            else:
                print("   ❌ Workspace URL não disponível")
        except Exception as e:
            print(f"   ❌ Erro ao obter workspace URL: {e}")
        
        # API Token
        try:
            api_token = context.apiToken().get() if context.apiToken().isDefined() else None
            if api_token:
                masked_token = f"{api_token[:15]}...{api_token[-8:]}" if len(api_token) > 23 else "***"
                print(f"   ✅ Token obtido: {masked_token}")
                # Armazenar para testes posteriores
                globals()['detected_token'] = api_token
            else:
                print("   ❌ Token não disponível")
        except Exception as e:
            print(f"   ❌ Erro ao obter token: {e}")
            
    except Exception as e:
        print(f"   ❌ Erro ao acessar contexto: {e}")
else:
    print("   ❌ dbutils não encontrado em globals")

# 2. Tentar importar DBUtils
print(f"\n2️⃣ Tentar importar DBUtils:")
try:
    from pyspark.dbutils import DBUtils
    from pyspark.sql import SparkSession
    
    spark = SparkSession.getActiveSession()
    if spark:
        dbutils_imported = DBUtils(spark)
        print("   ✅ DBUtils importado com sucesso")
        
        # Tentar extrair contexto
        try:
            context = dbutils_imported.notebook.entry_point.getDbutils().notebook().getContext()
            
            # API URL via import
            try:
                api_url = context.apiUrl().get() if context.apiUrl().isDefined() else None
                if api_url:
                    print(f"   ✅ Workspace URL (importado): {api_url}")
                else:
                    print("   ❌ Workspace URL não disponível (importado)")
            except Exception as e:
                print(f"   ❌ Erro ao obter workspace URL (importado): {e}")
            
            # API Token via import
            try:
                api_token = context.apiToken().get() if context.apiToken().isDefined() else None
                if api_token:
                    masked_token = f"{api_token[:15]}...{api_token[-8:]}" if len(api_token) > 23 else "***"
                    print(f"   ✅ Token obtido (importado): {masked_token}")
                else:
                    print("   ❌ Token não disponível (importado)")
            except Exception as e:
                print(f"   ❌ Erro ao obter token (importado): {e}")
                
        except Exception as e:
            print(f"   ❌ Erro ao acessar contexto (importado): {e}")
    else:
        print("   ❌ SparkSession não ativa")
        
except ImportError as e:
    print(f"   ❌ Erro ao importar DBUtils: {e}")
except Exception as e:
    print(f"   ❌ Erro geral: {e}")

# 3. Verificar Spark context
print(f"\n3️⃣ Verificar Spark context:")
try:
    if 'spark' in globals():
        spark_session = globals()['spark']
        spark_conf = spark_session.sparkContext.getConf()
        workspace_url = spark_conf.get('spark.databricks.workspaceUrl')
        
        if workspace_url:
            print(f"   ✅ Workspace URL (Spark): {workspace_url}")
        else:
            print("   ❌ spark.databricks.workspaceUrl não encontrada")
            
        # Listar todas as configurações do Spark
        print("   📋 Configurações Spark relevantes:")
        all_conf = spark_conf.getAll()
        databricks_conf = [conf for conf in all_conf if 'databricks' in conf[0].lower()]
        for key, value in databricks_conf[:5]:  # Mostrar apenas 5 primeiras
            print(f"      {key}: {value}")
    else:
        print("   ❌ Spark session não disponível em globals")
        
except Exception as e:
    print(f"   ❌ Erro ao verificar Spark: {e}")

# 4. Verificar variáveis de ambiente
print(f"\n4️⃣ Verificar variáveis de ambiente:")
env_vars = ['DATABRICKS_HOST', 'DATABRICKS_TOKEN', 'DATABRICKS_WORKSPACE_URL']
for var in env_vars:
    value = os.getenv(var)
    if value:
        if 'TOKEN' in var:
            masked = f"{value[:10]}...{value[-5:]}" if len(value) > 15 else "***"
            print(f"   ✅ {var}: {masked}")
        else:
            print(f"   ✅ {var}: {value}")
    else:
        print(f"   ❌ {var}: (não definido)")

# 5. Resumo final
print(f"\n📊 Resumo de Detecção:")
methods_found = []
if 'detected_workspace_url' in globals():
    methods_found.append("Workspace URL via dbutils")
if 'detected_token' in globals():
    methods_found.append("Token via dbutils")
if 'spark' in globals():
    methods_found.append("Spark context")

if methods_found:
    print(f"   ✅ Métodos disponíveis: {', '.join(methods_found)}")
    print(f"   🎯 Dino SDK v1.1.0 deve funcionar!")
else:
    print(f"   ❌ Nenhum método de detecção funcionou")
    print(f"   💡 Possível problema de configuração do ambiente")

## 2. 📦 Instalar e Testar Dino SDK v1.1.0

Instalar a nova versão com detecção de contexto aprimorada.

In [ ]:
# Instalar Dino SDK v1.1.0
print("📦 Instalando Dino SDK v1.1.0...")

# Caminho para a wheel (ajustar conforme necessário)
wheel_path = "/dbfs/path/to/dino_sdk-1.1.0-py3-none-any.whl"
# Alternativa local (ajustar caminho)
# wheel_path = "/Workspace/Shared/wheels/dino_sdk-1.1.0-py3-none-any.whl"

try:
    # Para notebooks Databricks, usar o caminho do DBFS
    %pip install /dbfs/path/to/dino_sdk-1.1.0-py3-none-any.whl --force-reinstall --quiet
    print("✅ Dino SDK v1.1.0 instalado via DBFS")
except:
    try:
        # Fallback para caminho local
        %pip install /Workspace/Shared/wheels/dino_sdk-1.1.0-py3-none-any.whl --force-reinstall --quiet  
        print("✅ Dino SDK v1.1.0 instalado via Workspace")
    except:
        print("❌ Falha na instalação - ajuste o caminho da wheel")
        print("💡 Carregue a wheel em /dbfs/ ou /Workspace/Shared/")

import time
time.sleep(3)

# Testar importação
try:
    from src.keyvault_config import KeyVaultConfigManager, get_notebook_context
    print("✅ Módulos importados com sucesso")
    print("   • KeyVaultConfigManager")
    print("   • get_notebook_context (v1.1.0)")
except ImportError as e:
    print(f"❌ Erro na importação: {e}")
    print("💡 Verifique se a wheel foi instalada corretamente")

## 3. 🔧 Testar Nova Função get_notebook_context

Vamos testar a função melhorada de detecção de contexto.

In [ ]:
# Testar função get_notebook_context aprimorada
from src.keyvault_config import get_notebook_context

print("🔧 Testando get_notebook_context v1.1.0")
print("=" * 45)

try:
    # Chamar função aprimorada
    context = get_notebook_context()
    
    if context:
        print("✅ Contexto extraído com sucesso!")
        
        # Verificar conteúdo do contexto
        print(f"\n📋 Conteúdo do contexto:")
        
        if context.get('dbutils'):
            print(f"   ✅ dbutils: Disponível")
        else:
            print(f"   ❌ dbutils: Não disponível")
            
        if context.get('workspace_url'):
            print(f"   ✅ workspace_url: {context['workspace_url']}")
        else:
            print(f"   ❌ workspace_url: Não obtida")
            
        if context.get('token'):
            masked_token = f"{context['token'][:15]}...{context['token'][-8:]}"
            print(f"   ✅ token: {masked_token}")
        else:
            print(f"   ❌ token: Não obtido")
        
        # Salvar contexto para próximos testes
        globals()['extracted_context'] = context
        
        # Status da extração
        completeness = sum([
            bool(context.get('dbutils')),
            bool(context.get('workspace_url')),
            bool(context.get('token'))
        ])
        
        print(f"\n📊 Completeness: {completeness}/3 elementos extraídos")
        
        if completeness >= 2:
            print(f"🎉 SUCESSO! Contexto suficiente para inicialização")
        elif completeness == 1:
            print(f"⚠️ PARCIAL - Pode necessitar fallbacks")
        else:
            print(f"❌ FALHA - Contexto insuficiente")
            
    else:
        print("❌ Nenhum contexto foi extraído")
        print("💡 Possíveis causas:")
        print("   • dbutils não disponível no ambiente")
        print("   • Spark session não ativa")
        print("   • Ambiente não é notebook Databricks")
        
except Exception as e:
    print(f"❌ Erro ao testar get_notebook_context: {e}")
    import traceback
    traceback.print_exc()

## 4. 🚀 Testar Inicialização Completa

Vamos testar a inicialização completa do KeyVaultConfigManager com a nova detecção de contexto.

In [ ]:
# Testar inicialização completa com v1.1.0
print("🚀 Testando Inicialização Completa - Dino SDK v1.1.0")
print("=" * 55)

# Criar KeyVaultConfigManager
try:
    kv_manager = KeyVaultConfigManager(
        keyvault_name="dino-shared-keyvault",
        catalog_name="dino_catalog",
        schema_name="teste_context_v110"
    )
    
    print(f"✅ KeyVaultConfigManager criado")
    print(f"🔑 Secret Scope: {kv_manager.SECRET_SCOPE_NAME}")
    
    # Testar inicialização do cliente Databricks
    print(f"\n🔧 Iniciando cliente Databricks...")
    
    # A v1.1.0 deve usar a nova detecção de contexto
    kv_manager._initialize_databricks_client()
    
    # Verificar resultado
    if hasattr(kv_manager, 'databricks_client') and kv_manager.databricks_client:
        print(f"✅ Cliente Databricks inicializado com sucesso!")
        
        # Verificar método usado
        if hasattr(kv_manager, '_temporary_token'):
            print(f"🔑 Método usado: Token temporário")
            masked_token = f"{kv_manager._temporary_token[:10]}...{kv_manager._temporary_token[-8:]}"
            print(f"   Token: {masked_token}")
        else:
            print(f"🔑 Método usado: Contexto direto do notebook")
        
        # Testar operações básicas
        try:
            current_user = kv_manager.databricks_client.current_user.me()
            print(f"👤 Usuário conectado: {current_user.user_name}")
            print(f"📧 Email: {current_user.emails[0].value if current_user.emails else 'N/A'}")
            
            # Testar acesso a Secret Scopes
            try:
                scopes = kv_manager.databricks_client.secrets.list_scopes()
                scope_names = [scope.name for scope in scopes]
                print(f"📋 Secret Scopes acessíveis: {len(scope_names)}")
                
                # Verificar se nossa scope existe
                target_scope = kv_manager.SECRET_SCOPE_NAME
                if target_scope in scope_names:
                    print(f"✅ Scope '{target_scope}' encontrada")
                else:
                    print(f"⚠️ Scope '{target_scope}' não existe (será criada)")
                    
            except Exception as e:
                print(f"⚠️ Erro ao listar Secret Scopes: {e}")
                print("💡 Pode ser problema de permissões")
                
        except Exception as e:
            print(f"❌ Erro ao testar operações básicas: {e}")
            
    else:
        print(f"❌ Cliente Databricks não foi inicializado")
        print(f"💡 Verifique os logs de inicialização acima")
        
except Exception as e:
    print(f"❌ Erro durante inicialização: {e}")
    import traceback
    traceback.print_exc()

## 5. 🔍 Comparação: v1.1.0 vs Versões Anteriores

Vamos comparar os métodos de detecção entre versões.

In [ ]:
# Comparação entre métodos de detecção
print("🔍 Comparação: Métodos de Detecção")
print("=" * 40)

print("📊 Evolução dos Métodos:")

print(f"\n🔴 Versões Antigas (< 1.0.9):")
print(f"   ❌ Dependia apenas de dbutils em globals")
print(f"   ❌ Falha se dbutils não estava disponível")
print(f"   ❌ Sem extração de token do contexto")

print(f"\n🟡 Versão 1.0.9:")
print(f"   ✅ Adicionou criação de token temporário")
print(f"   ✅ Múltiplos métodos Azure de auth")
print(f"   ⚠️ Ainda dependia de variáveis de ambiente para workspace URL")

print(f"\n🟢 Versão 1.1.0 (ATUAL):")
print(f"   ✅ Extração nativa de token via dbutils context")
print(f"   ✅ Detecção automática de workspace URL")
print(f"   ✅ Fallbacks inteligentes (globals → import → spark → env)")
print(f"   ✅ Logs detalhados para diagnóstico")

# Demonstrar diferenças práticas
print(f"\n🧪 Teste Comparativo:")

# Método antigo (simulado)
old_method_success = False
if 'dbutils' in globals():
    print(f"   ✅ Método antigo: dbutils disponível em globals")
    old_method_success = True
else:
    print(f"   ❌ Método antigo: dbutils não em globals - FALHARIA")

# Método novo (v1.1.0)
new_method_success = False
context = get_notebook_context()
if context and (context.get('workspace_url') or context.get('token')):
    print(f"   ✅ Método novo (v1.1.0): Contexto extraído com sucesso")
    new_method_success = True
else:
    print(f"   ❌ Método novo (v1.1.0): Falhou na extração")

# Resultado
if new_method_success and not old_method_success:
    print(f"\n🎉 MELHORIA CONFIRMADA!")
    print(f"   v1.1.0 funciona onde versões antigas falhariam")
elif new_method_success and old_method_success:
    print(f"\n✅ COMPATIBILIDADE MANTIDA")
    print(f"   v1.1.0 mantém funcionalidade + melhorias")
else:
    print(f"\n⚠️ AMBIENTE DESAFIADOR")
    print(f"   Ambiente pode precisar configuração adicional")

# Benefícios específicos da v1.1.0
print(f"\n🚀 Benefícios Específicos v1.1.0:")
beneficios = [
    "Extração direta de token do notebook (sem Azure auth externa)",
    "Auto-detecção de workspace URL (sem variáveis de ambiente)",
    "Múltiplos métodos de fallback (robustez aumentada)",
    "Logs detalhados (melhor debugging)",
    "Compatibilidade total com versões anteriores"
]

for i, beneficio in enumerate(beneficios, 1):
    print(f"   {i}. {beneficio}")

## 6. ✅ Validação Final e Status

Resumo dos testes e recomendações para uso.

In [ ]:
# Validação final v1.1.0
print("✅ Validação Final - Dino SDK v1.1.0")
print("=" * 42)

# Executar testes de validação
tests = {
    'context_detection': False,
    'workspace_url': False, 
    'token_extraction': False,
    'client_initialization': False,
    'basic_operations': False
}

# 1. Detecção de contexto
context = get_notebook_context()
if context:
    tests['context_detection'] = True
    print(f"✅ Detecção de contexto: Funcionando")
    
    # 2. Workspace URL
    if context.get('workspace_url'):
        tests['workspace_url'] = True
        print(f"✅ Workspace URL: {context['workspace_url']}")
    else:
        print(f"❌ Workspace URL: Não detectada")
    
    # 3. Token
    if context.get('token'):
        tests['token_extraction'] = True
        print(f"✅ Token: Extraído do contexto")
    else:
        print(f"❌ Token: Não extraído")
else:
    print(f"❌ Detecção de contexto: Falhou")

# 4. Inicialização do cliente
if 'kv_manager' in globals() and hasattr(kv_manager, 'databricks_client') and kv_manager.databricks_client:
    tests['client_initialization'] = True
    print(f"✅ Cliente Databricks: Inicializado")
    
    # 5. Operações básicas
    try:
        current_user = kv_manager.databricks_client.current_user.me()
        tests['basic_operations'] = True
        print(f"✅ Operações básicas: Funcionando ({current_user.user_name})")
    except Exception as e:
        print(f"❌ Operações básicas: Falharam ({e})")
else:
    print(f"❌ Cliente Databricks: Não inicializado")

# Calcular score
passed = sum(tests.values())
total = len(tests)
score_percent = (passed / total) * 100

print(f"\n📊 Score de Validação: {passed}/{total} ({score_percent:.0f}%)")

# Status final
if score_percent >= 80:
    print(f"\n🎉 EXCELENTE! Dino SDK v1.1.0 está funcionando perfeitamente")
    status = "PRONTO PARA PRODUÇÃO"
elif score_percent >= 60:
    print(f"\n🎯 BOM! Dino SDK v1.1.0 está funcional com pequenos ajustes")
    status = "FUNCIONAL COM RESSALVAS"
elif score_percent >= 40:
    print(f"\n⚠️ PARCIAL. Algumas funcionalidades precisam de configuração")
    status = "NECESSITA CONFIGURAÇÃO"
else:
    print(f"\n❌ PROBLEMÁTICO. Verifique configuração do ambiente")
    status = "REQUER TROUBLESHOOTING"

# Próximos passos baseados no status
print(f"\n🚀 Próximos Passos ({status}):")

if score_percent >= 80:
    print(f"   1. Execute setup completo: dino-config setup --keyvault dino-shared-keyvault")
    print(f"   2. Configure seu projeto específico")
    print(f"   3. Inicie ingestão de dados")
elif score_percent >= 60:
    print(f"   1. Verifique permissões do usuário no workspace")
    print(f"   2. Configure Secret Scopes se necessário")
    print(f"   3. Execute testes em ambiente de produção")
elif score_percent >= 40:
    print(f"   1. Configure variáveis de ambiente DATABRICKS_HOST/TOKEN")
    print(f"   2. Verifique se o cluster tem permissões adequadas")
    print(f"   3. Execute 'az login' se usando autenticação Azure")
else:
    print(f"   1. Verifique se está executando em notebook Databricks")
    print(f"   2. Confirme que o cluster está ativo e acessível")
    print(f"   3. Contacte suporte para troubleshooting avançado")

# Informações da versão
print(f"\n📋 Informações da Versão:")
print(f"   🦕 Dino SDK: v1.1.0")
print(f"   🔧 Funcionalidade: Detecção de contexto aprimorada")
print(f"   🎯 Foco: Extração nativa de token e workspace URL")
print(f"   📈 Score: {score_percent:.0f}% de funcionalidade validada")
print(f"   🔒 Status: {status}")